# Armaduras 2D con `armadura-2d-andresdperezc`

Notebook didáctico: un poco de teoría de mecánica vectorial y llamadas a la librería. **No usa Streamlit.**

Instalación previa, desde la carpeta del proyecto (use el kernel de este `.venv`):

```powershell
python -m venv .venv
.\.venv\Scripts\Activate.ps1
python -m pip install -e ".[dev]"
```

Cuando el paquete esté en PyPI: `pip install armadura-2d-andresdperezc`.

## 1. Qué es una armadura 2D ideal

Una **armadura** (puentes, techos) está hecha de barras unidas en nudos por pasadores. Cada barra es un elemento de **dos fuerzas**: solo axial (tensión o compresión).

En cada nudo:

$$\sum F_x = 0, \qquad \sum F_y = 0$$

La geometría de una barra $i\to j$ es

$$L = \sqrt{\Delta x^2 + \Delta y^2}, \quad \cos\theta = \Delta x / L, \quad \sin\theta = \Delta y / L.$$

Python arma el sistema $A\,x = b$: $A$ son senos y cosenos, $x$ las fuerzas internas y las reacciones, $b$ las cargas. Signo: **positivo = tensión**, **negativo = compresión**.

Isostática 2D: $m + r = 2j$ (barras + reacciones = dos ecuaciones por nudo). Este paquete **no** resuelve hiperestáticas ni pórticos.

## 2. Importar y ver el preset de libro

Unidades: **kN** y **m**. Eje $y$ positivo hacia arriba. Una carga de 50 kN hacia abajo es $F_y = -50$.

In [ ]:
from armadura_2d_andresdperezc import Armadura2D

print(Armadura2D.TIPOS_APOYO)
print("tolerancia equilibrio (kN) =", Armadura2D.TOLERANCIA_EQUILIBRIO)

triangulo = Armadura2D.preset_triangulo()
print(triangulo)
print("L_B1 =", triangulo.longitud("B1"), "m")

## 3. Caso de libro: triángulo 50 kN

Nudos $(0,0)$ fijo, $(4,0)$ móvil, $(2,3)$ con $50\,\mathrm{kN}$ hacia abajo.  
Libro: $R_y = 25\,\mathrm{kN}$ en cada apoyo; B1 $= +50/3\,\mathrm{kN}$ (tensión); B2 y B3 $= -(25/3)\sqrt{13}\,\mathrm{kN}$ (compresión).

In [ ]:
resultado = triangulo.resolver()
print("equilibrio ok:", resultado.equilibrio.ok)
for rec in resultado.reacciones:
    print(f"nudo {rec.nudo}: Rx = {rec.rx:.3f} kN, Ry = {rec.ry:.3f} kN")
print()
for barra in resultado.fuerzas:
    print(f"{barra.id}: N = {barra.fuerza:8.3f} kN  {barra.tipo}")

## 4. Construir a mano (sin GUI)

La librería acepta nudos, barras, apoyos y cargas. Streamlit, cuando exista, solo llamará a esta misma API.

In [ ]:
manual = Armadura2D(nombre="manual")
manual.agregar_nodo(0, 0.0, 0.0)
manual.agregar_nodo(1, 4.0, 0.0)
manual.agregar_nodo(2, 2.0, 3.0)
manual.agregar_barra("B1", 0, 1)
manual.agregar_barra("B2", 0, 2)
manual.agregar_barra("B3", 1, 2)
manual.agregar_apoyo(0, "fijo")
manual.agregar_apoyo(1, "movil")
manual.agregar_carga(2, fx=0.0, fy=-50.0)
print(manual.resolver().fuerzas[0].fuerza)

## 5. Pratt vs Howe

Misma luz 4 m × 2 m y la misma carga en el tablero. En Pratt las diagonales apuntan al centro y van a **tracción**; en Howe apuntan a los apoyos y van a **compresión**.

In [ ]:
def entre(resultado, i, j):
    par = frozenset((i, j))
    return next(
        b for b in resultado.fuerzas if frozenset((b.nudo_i, b.nudo_j)) == par
    )

pratt = Armadura2D.preset_pratt().resolver()
howe = Armadura2D.preset_howe().resolver()
print("Pratt 3–1:", entre(pratt, 3, 1).tipo, f"{entre(pratt, 3, 1).fuerza:.3f} kN")
print("Howe  4–0:", entre(howe, 4, 0).tipo, f"{entre(howe, 4, 0).fuerza:.3f} kN")
print("Warren equilibrio:", Armadura2D.preset_warren().resolver().equilibrio.ok)

## 6. Gráficos

`graficar` **devuelve** una `Figure` de matplotlib; no abre una ventana Tkinter. En Jupyter la figura se muestra sola.

In [ ]:
triangulo.graficar(resultado)

In [ ]:
Armadura2D.preset_pratt().graficar()

## 7. Limitaciones

Solo 2D isostáticas, cargas en nudos, apoyo fijo + móvil vertical. Sin diámetro/esfuerzo, sin pórticos, sin 3D. Si el modelo se expande, hay que **volver a validarlo**.